# 1. Cost-Sensitive Learning: Handling Class Imbalance with `class_weight`

This notebook covers:
1. **The Imbalance Trap & The Accuracy Paradox**: Why standard models achieve 95%+ accuracy while failing completely on the minority class.
2. **The Mathematics of `class_weight='balanced'`**: How Scikit-Learn scales loss gradients proportionally to class frequencies.
3. **Training Baseline vs. Cost-Sensitive Models**: Comparing unweighted and balanced models across `LogisticRegression` and `RandomForestClassifier`.
4. **Custom Cost Matrices (`class_weight={0: 1, 1: 10}`)**: Tuning penalty ratios to reflect asymmetric business costs.
5. **Proper Evaluation**: Focusing on Confusion Matrix, Recall, Precision, and F1-Score instead of raw Accuracy.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Generate a synthetic fraud detection dataset with a 95:5 class imbalance
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.95, 0.05],  # 95% Class 0 (Legit), 5% Class 1 (Fraud)
    flip_y=0,
    random_state=42
)

# Convert to DataFrame
feature_names = [f"Feature_{i+1}" for i in range(X_raw.shape[1])]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Is_Fraud'] = y_raw

print("=== 1. CLASS DISTRIBUTION ===")
display(df['Is_Fraud'].value_counts(normalize=True).rename('Proportion').to_frame().assign(
    Count=df['Is_Fraud'].value_counts()
))

=== 1. CLASS DISTRIBUTION ===


,Proportion,Count
Is_Fraud,,
0,0.95,950
1,0.05,50


---
## Part 1: Train / Test Split (Stratified)

We perform an 80/20 train/test split.
* **`stratify=y` is mandatory**: Ensures both the training and test sets contain the exact same 95:5 class distribution.